# Hito 3 — Análisis Exploratorio de Datos (EDA)

**Proyecto Integrador — Minería de Datos 1**

Trabajamos sobre el dataset ya limpio (`data/processed/streaming_users_clean.csv`, 8.000 filas).

## Preguntas de análisis
Antes de graficar, definimos qué queremos responder con evidencia:

1. ¿El plan de suscripción está asociado a un mayor consumo mensual de contenido?
2. ¿La cantidad de tickets de soporte varía según el plan contratado?
3. ¿Existe relación entre la edad de un usuario y su tiempo de consumo?
4. ¿La distribución de planes de suscripción es similar en todos los países?
5. De las variables numéricas disponibles (edad, consumo, tickets), ¿cuál está más asociada al plan de suscripción?

Cada gráfico de esta sección apunta a responder una de estas preguntas.


## 0. Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

df = pd.read_csv("../data/processed/streaming_users_clean.csv", parse_dates=["last_login_date"])
df.head()


## 1. Análisis univariado

### 1.1 Distribución de la edad

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(df["age"].dropna(), bins=30, color="steelblue")
plt.title("Distribución de la edad de los usuarios")
plt.xlabel("Edad")
plt.ylabel("Cantidad de usuarios")
plt.show()


**Interpretación:** la mayoría de los usuarios tiene entre 25 y 42 años (edad media ≈ 33.5 años),
con una distribución relativamente simétrica. Se observa un pequeño grupo de edades muy bajas
(0 y 4 años, 46 registros en total) que no encajan con el resto de la distribución, que arranca
de forma natural recién en los 13 años. Es poco probable que existan cuentas contratadas por bebés
o niños de esa edad como titulares; lo tratamos como una limitación conocida del dataset más que
como un hallazgo, ya que no identificamos una regla clara para separarlos como error (a diferencia
de los valores sentinela detectados en el Hito 2).

### 1.2 Distribución del tiempo de consumo mensual

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(df["monthly_watch_time_mins"].dropna(), bins=30, color="darkorange")
plt.title("Distribución de minutos de consumo mensual")
plt.xlabel("Minutos de consumo mensual")
plt.ylabel("Cantidad de usuarios")
plt.show()


**Interpretación:** el consumo mensual promedio es de aproximadamente 795 minutos (~13 horas al
mes), con una mediana muy similar (757 minutos), lo que indica una distribución razonablemente
simétrica y sin grandes outliers una vez removidos los valores sentinela en el Hito 2. Esto da una
base confiable para comparar el consumo entre distintos grupos de usuarios (pregunta 1 y 3).

### 1.3 Distribución de planes de suscripción

In [ ]:
plt.figure(figsize=(6,4))
orden = df["subscription_plan"].value_counts().index
sns.countplot(data=df, x="subscription_plan", order=orden, color="mediumseagreen")
plt.title("Cantidad de usuarios por plan de suscripción")
plt.xlabel("Plan")
plt.ylabel("Cantidad de usuarios")
plt.show()


**Interpretación:** el plan **Básico** es el más elegido (3.600 usuarios, 45%), seguido de
**Estándar** (2.817, 35%) y **Premium** (1.583, 20%). Es una distribución esperable para una
plataforma de streaming, donde el plan de entrada suele concentrar a la mayoría de la base.

### 1.4 Distribución de tickets de soporte

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="customer_support_tickets", color="indianred")
plt.title("Cantidad de tickets de soporte por usuario")
plt.xlabel("Tickets de soporte")
plt.ylabel("Cantidad de usuarios")
plt.show()


**Interpretación:** la gran mayoría de los usuarios generó 0 o 1 ticket de soporte (más del 80%
de la base), y son pocos los que llegan a 4 o 5. Es una variable con poca dispersión, lo que
anticipa que difícilmente explique diferencias grandes entre grupos (lo confirmamos en la
pregunta 2).

## 2. Análisis bivariado

### 2.1 Consumo mensual según plan de suscripción (Pregunta 1)

In [ ]:
plt.figure(figsize=(7,4))
orden = ["Básico","Estándar","Premium"]
sns.boxplot(data=df, x="subscription_plan", y="monthly_watch_time_mins", order=orden, palette="Blues")
plt.title("Consumo mensual según plan de suscripción")
plt.xlabel("Plan")
plt.ylabel("Minutos de consumo mensual")
plt.show()


**Interpretación:** hay una relación clara y creciente entre el plan contratado y el consumo
mensual: el promedio pasa de 597 minutos en Básico, a 872 en Estándar, a 1.140 en Premium. Esto
responde la pregunta 1: sí, el plan de suscripción está asociado a un mayor consumo, lo cual es
coherente (usuarios que consumen más contenido tienden a optar por planes superiores, o los
planes superiores habilitan mayor consumo).

### 2.2 Tickets de soporte según plan de suscripción (Pregunta 2)

In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(data=df, x="subscription_plan", y="customer_support_tickets", order=orden, palette="Reds", errorbar=None)
plt.title("Promedio de tickets de soporte según plan")
plt.xlabel("Plan")
plt.ylabel("Tickets de soporte (promedio)")
plt.show()


**Interpretación:** el promedio de tickets es prácticamente el mismo en los tres planes
(entre 0.79 y 0.81). Esto responde la pregunta 2: el plan contratado **no** parece influir en la
cantidad de tickets de soporte generados; los problemas de soporte parecen ser independientes del
nivel de suscripción.

### 2.3 Edad vs. consumo mensual (Pregunta 3)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x="age", y="monthly_watch_time_mins", alpha=0.3, color="teal")
plt.title("Relación entre edad y consumo mensual")
plt.xlabel("Edad")
plt.ylabel("Minutos de consumo mensual")
plt.show()

print("Correlación edad-consumo:", round(df["age"].corr(df["monthly_watch_time_mins"]), 4))


**Interpretación:** la nube de puntos no muestra ninguna tendencia visible, y la correlación es
prácticamente nula (≈ 0.004). Esto responde la pregunta 3: en este dataset, la edad **no** explica
el tiempo de consumo; usuarios de todas las edades consumen cantidades similares de contenido, en
promedio.

### 2.4 Distribución de planes por país (Pregunta 4)

In [ ]:
tabla = pd.crosstab(df["country"], df["subscription_plan"], normalize="index")[orden] * 100

tabla.plot(kind="bar", stacked=True, figsize=(8,5), color=["#a1c9f4","#8de5a1","#ff9f9b"])
plt.title("Distribución porcentual de planes por país")
plt.xlabel("País")
plt.ylabel("% de usuarios")
plt.legend(title="Plan")
plt.xticks(rotation=30)
plt.show()

tabla.round(1)


**Interpretación:** la proporción de cada plan es muy similar en los 7 países (siempre
alrededor de 44-46% Básico, 34-37% Estándar y 19-21% Premium). Esto responde la pregunta 4: el
país **no** parece influir en el plan elegido; la composición de la base es prácticamente
uniforme a nivel regional.

## 3. Análisis multivariado

### 3.1 Correlación entre variables numéricas

In [ ]:
plt.figure(figsize=(5,4))
corr = df[["age","monthly_watch_time_mins","customer_support_tickets"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlación entre variables numéricas")
plt.show()


**Interpretación:** ninguna de las tres variables numéricas está correlacionada entre sí
(todos los valores son cercanos a 0). Esto es un hallazgo relevante para el PCA del Hito 4: si
las variables no están correlacionadas, la reducción de dimensionalidad va a tener menos margen
para resumir información en pocas componentes, ya que cada variable aporta información
relativamente independiente.

### 3.2 Edad, consumo y tickets combinados según plan (Pregunta 5)

In [ ]:
resumen = df.groupby("subscription_plan")[["age","monthly_watch_time_mins","customer_support_tickets"]].mean().loc[orden]
resumen.round(1)


In [ ]:
resumen_norm = resumen / resumen.max()

resumen_norm.plot(kind="bar", figsize=(8,5))
plt.title("Comparación normalizada de variables numéricas por plan")
plt.ylabel("Valor relativo al máximo de cada variable")
plt.xlabel("Plan")
plt.xticks(rotation=0)
plt.legend(title="Variable")
plt.show()


**Interpretación:** al comparar las tres variables juntas por plan, se ve claramente que
**el consumo mensual es la única variable que varía de forma marcada entre planes**, mientras que
la edad y los tickets de soporte se mantienen prácticamente constantes. Esto responde la
pregunta 5: de las tres variables numéricas disponibles, el consumo mensual es la que mejor
diferencia a los usuarios según su plan de suscripción.

## 4. Resumen del Hito 3

| Pregunta | Respuesta (con evidencia) |
|---|---|
| ¿El plan está asociado a mayor consumo? | Sí: el consumo promedio crece de 597 (Básico) a 1.140 min (Premium) |
| ¿Los tickets varían según el plan? | No: el promedio es ≈0.8 en los tres planes |
| ¿La edad se relaciona con el consumo? | No: correlación ≈ 0.004, sin tendencia visible |
| ¿La distribución de planes varía por país? | No: proporciones similares (~45/35/20%) en los 7 países |
| ¿Qué variable numérica se asocia más al plan? | El consumo mensual, claramente por encima de edad y tickets |

**Limitación detectada:** un pequeño grupo de usuarios con edades de 0 y 4 años (46 registros) no
encaja con el resto de la distribución de edad, pero no lo tratamos como error porque no hay una
regla clara para distinguirlo de datos válidos, a diferencia de los valores sentinela tratados en
el Hito 2. Queda documentado como limitación para las conclusiones finales (Hito 5).

Estos hallazgos, en particular que el consumo mensual es la variable numérica más relevante y que
las tres variables numéricas no están correlacionadas entre sí, van a ser el punto de partida para
el escalamiento y PCA del Hito 4 (`04_pca.ipynb`).
